In [ ]:
import snowflake.connector
print(snowflake.connector.__version__)
from snowflake.connector.pandas_tools import write_pandas

4.7.3


In [4]:
import configparser
import os

config = configparser.ConfigParser()

# config.ini is in the same folder as the notebook
config_path = os.path.join(os.getcwd(), "config.ini")

if not os.path.exists(config_path):
    raise FileNotFoundError(f"config.ini not found: {config_path}")

config.read(config_path)

if "SNOWFLAKE" not in config:
    raise KeyError(
        f"[SNOWFLAKE] section not found in {config_path}. "
        f"Available sections: {config.sections()}"
    )

user = config["SNOWFLAKE"]["user"]
password = config["SNOWFLAKE"]["password"]
account = config["SNOWFLAKE"]["account"]
warehouse = config["SNOWFLAKE"]["warehouse"]
database = config["SNOWFLAKE"]["database"]
schema = config["SNOWFLAKE"]["schema"]

print("Configuration loaded successfully.")

Configuration loaded successfully.


# Establishing Snowflake connection

In [10]:
try:
    conn = snowflake.connector.connect(
        user=user,
        password=password,
        account=account,
        warehouse=warehouse,
        database=database,
    schema=schema)
    print("Connection to Snowflake established successfully.")
except Exception as e:
    print(f"Failed to connect to Snowflake: {e}")
    raise

Connection to Snowflake established successfully.


# Reading Data from Local

In [6]:
PATH=r"F:\all data from customersite.csv"

In [8]:
import pandas as pd

In [ ]:
data = pd.read_csv(PATH)
data.head()

In [ ]:
### Date columns to be converted to datetime format
date_cols =['initial_active_date','InitialContractStart', 'InitialContractEnd','InitialContractStatusEffectiveDate',
       'InitialContractCancelEffectiveDate', 'recent_active_date','RecentContractStart', 'RecentContractEnd', 'RecentContractStatusEffectiveDate','RecentContractCancelEffectiveDate', 'cancel_date_new', 'profile_date']

In [ ]:
data[date_cols] = data[date_cols].apply(
    lambda col: pd.to_datetime(col, errors='coerce').dt.normalize()
)

In [20]:
data.head()

,CustomerNumber,SiteNumber,initial_active_date,InitialContractStart,InitialContractEnd,InitialContractInitialMonths,InitialContractRenewalMonths,InitialContractRenewals,InitialContractStatusEffectiveDate,InitialContractCancelEffectiveDate,...,RecentContractEnd,RecentContractInitialMonths,RecentContractRenewalMonths,RecentContractRenewals,RecentContractStatusEffectiveDate,RecentContractCancelEffectiveDate,cancel_date_new,profile_date,account_status,property_type
0,3720.0,900102532.0,2003-04-10,2018-06-01,2019-05-31,36.0,12.0,13.0,2019-01-29 07:54:00,2019-01-31,...,2019-05-31,36.0,12.0,13.0,2019-01-29 07:54:00,2019-01-31,2019-01-28 16:14:00,2025-09-01,Cancelled,Residential
1,3721.0,900103917.0,1999-05-24,2001-06-01,2009-04-30,0.0,0.0,0.0,2009-04-08 12:13:00,2009-04-30,...,2009-04-30,0.0,0.0,0.0,2009-04-08 12:13:00,2009-04-30,2009-04-30 11:25:00,2025-09-01,Cancelled,Commercial
2,3722.0,900025588.0,1999-05-24,2001-06-01,2005-10-31,0.0,0.0,0.0,2006-03-30 16:43:00,2005-10-31,...,2005-10-31,0.0,0.0,0.0,2006-03-30 16:43:00,2005-10-31,2006-03-02 09:22:00,2025-09-01,Cancelled,Commercial
3,3727.0,900012900.0,1999-05-14,2002-08-01,2011-08-31,0.0,0.0,0.0,2011-09-14 12:31:00,2011-08-31,...,2026-03-31,24.0,12.0,4.0,2021-04-02 00:00:00,NaT,NaT,2025-09-01,Active,Residential
4,3729.0,900012901.0,1999-05-20,2008-05-21,2009-05-20,36.0,12.0,3.0,2008-08-13 14:10:00,2008-08-31,...,2009-05-20,36.0,12.0,3.0,2008-08-13 14:10:00,2008-08-31,2008-10-27 13:08:00,2025-09-01,Cancelled,Residential


#### Uploading data to Snowflake

In [ ]:
data.columns = data.columns.str.upper()

success, nchunks, nrows, output = write_pandas(
    conn=conn,
    df=data,
    table_name="CUSTOMERSITE",
    auto_create_table=True,
    overwrite=True
)

print(f"Success: {success}")
print(f"Rows uploaded: {nrows}")

Success: True
Rows uploaded: 875628


#### Read the data back from Snowflake to verify the upload

In [14]:
query = "SELECT * FROM CUSTOMERSITE_CLEAN LIMIT 5"

In [15]:
data = conn.cursor().execute(query).fetch_pandas_all()
data.head()

,CUSTOMERNUMBER,SITENUMBER,INITIALCONTRACTINITIALMONTHS,INITIALCONTRACTRENEWALMONTHS,INITIALCONTRACTRENEWALS,RECENTCONTRACTINITIALMONTHS,RECENTCONTRACTRENEWALMONTHS,RECENTCONTRACTRENEWALS,ACCOUNT_STATUS,PROPERTY_TYPE,...,INITIALCONTRACTEND,INITIALCONTRACTSTATUSEFFECTIVEDATE,INITIALCONTRACTCANCELEFFECTIVEDATE,RECENT_ACTIVE_DATE,RECENTCONTRACTSTART,RECENTCONTRACTEND,RECENTCONTRACTSTATUSEFFECTIVEDATE,RECENTCONTRACTCANCELEFFECTIVEDATE,CANCEL_DATE_NEW,PROFILE_DATE
0,8971459.0,100358276.0,36.0,12.0,2.0,36.0,12.0,2.0,Cancelled,Residential,...,2022-02-28,2021-04-02,2021-04-30,2017-03-04,2021-03-01,2022-02-28,2021-04-02,2021-04-30,2021-03-11,2025-09-01
1,8971475.0,100358279.0,36.0,12.0,4.0,36.0,12.0,4.0,Cancelled,Commercial,...,2024-03-31,2023-10-04,2023-10-31,2017-03-06,2023-04-01,2024-03-31,2023-10-04,2023-10-31,2023-09-22,2025-09-01
2,8971475.0,100359172.0,36.0,12.0,3.0,36.0,12.0,3.0,Cancelled,Residential,...,2023-03-31,2023-02-10,2023-02-28,2017-03-15,2022-04-01,2023-03-31,2023-02-10,2023-02-28,2023-02-10,2025-09-01
3,8971483.0,100358280.0,36.0,12.0,3.0,36.0,12.0,3.0,Cancelled,Residential,...,2023-03-31,2023-01-03,2023-01-31,2017-03-07,2022-04-01,2023-03-31,2023-01-03,2023-01-31,2022-12-16,2025-09-01
4,8971483.0,100645213.0,36.0,12.0,1.0,36.0,12.0,1.0,Active,Residential,...,None,2023-03-01,None,2023-03-14,2026-04-01,2026-03-31,2023-03-01,None,None,2025-09-01
